# Notebook06 — Mathematical improvements, three-hour experiment

Run All launches a real detached experiment on GPU0. The original three-hour deadline includes the attention audit, timing calibration, exact-resume checks, training and evaluation. The notebook returns while the worker continues in tmux.

**Question:** Can key centering, disjoint hierarchical semantic summaries and a fixed-budget feature allocation improve our semantic model's learning, step time or peak memory?

**Comparisons:** ViT; current stable semantic attention; centered tree attention with the training-only audit's chosen allocation. Paired initialization/data/augmentation/masks, effective batch128, FP32, 128×128images, 10,100pretraining images. Prefer3training seeds; use2only if timing cannot fit at least300updates perarm for3seeds. Freeze the common schedule after calibration, before new quality outcomes. Each initial/final encoder receives the same fixed20epoch probe on10,100labeled training images and2,020development images.

**Expected runtime:** at most3hours after first launch. This is a bounded exploratory comparison, not a guarantee that an accuracy difference will be statistically resolved. Equal updates/exposures do not mean equal compute. Official test remains untouched.

**Outputs:** `artifacts/notebook_06/<RUN_ID>/`, including the audit, timing, registered plan, recovery certificates, status, logs and stage report. **Checkpoints:** `checkpoints/<RUN_ID>/`; every50updates, epoch/stage boundary or15minutes, with full optimizer/scheduler/teacher/RNG/sampler state, latest two valid and best retained.

**Exact resume:** restart this notebook's kernel and Run All with the same RUN_ID and unchanged sources before its original deadline. It reconnects to a live worker or resumes verified checkpoints. Invalid state fails explicitly. After the original deadline, this run cannot be extended by Run All.

**Storage:** persistent boot disk; deletion policy unverified. Verify an off-instance copy of final/best checkpoints before instance deletion or recycling.

## 1. Load tagged definitions

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat, importlib.util, sys
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa").resolve()
_boot = {}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)
_spec = importlib.util.spec_from_loader("fmm06_workflow", _boot["TaggedNotebookLoader"](ROOT / "experiments/06_math/workflow.ipynb"))
WORKFLOW = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = WORKFLOW
_spec.loader.exec_module(WORKFLOW)


## 2. Registered bounded configuration

In [ ]:
RUN_ID = "math-improvements-001"
CONFIG = WORKFLOW.study_configuration(RUN_ID)
CONFIG


## 3. Launch or reconnect
This cell starts the actual worker. It verifies the operator certificate and binds source hashes before launch. A separate tmux watcher enforces the original deadline.

In [ ]:
LAUNCH = WORKFLOW.launch_study(ROOT, CONFIG)
LAUNCH


## 4. Inspect progress
Rerun this cell to refresh status. To attach: `tmux attach -t fmm-math-math-improvements-001`. Detach with Ctrl-b, then d; training continues. The final report is under `stages/learning/stage_results.md` in the artifact folder.

In [ ]:
STATUS = WORKFLOW.inspect_study(ROOT, RUN_ID)
STATUS


## Interpretation
Correctness requires operator checks, exact recovery and zero failed numerical attempts. Learning is measured against each model’s own initialization. Report candidate-minus-current and candidate-minus-ViT paired-seed accuracy/gain differences with uncertainty, plus measured time and memory ratios. An interval spanning zero remains unresolved; it does not establish equivalence. If the deadline cuts off required work, retain INCOMPLETE and the valid checkpoints.